In [ ]:
import earthaccess
import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
import matplotlib
import mpl_scatter_density

earthaccess.login()

Find sites with correct vegetation type

Granules have a naming pattern that includes the site name, vegetation type, and a description of the kind of data in the granule. We are interested in:
 - Deciduous broadleaf/needleleaf forest (DB, DN)
 - Evergreen broadleaf/needleleaf forest (EB, EN)
 - Grassland (GR)
 - Shrubs (SH)
 - Forest understory (UN)?

In [ ]:
all_site_meta = earthaccess.search_data(
    short_name="PhenoCam_V3_2389",
    granule_name="*meta.json"
)

In [ ]:
veg_types = ["DB", "DN", "EB", "EN", "GR", "SH", "UN"]

In [ ]:
def read_site_meta(f):
    return pd.read_json(f)["phenocam_site"].transpose()
try:
    site_meta = pd.read_parquet("phenocam_site_meta.parquet")
except FileNotFoundError:
    site_meta = pd.DataFrame(map(read_site_meta, earthaccess.open(all_site_meta))).set_index("sitename")
    site_meta.to_parquet("phenocam_site_meta.parquet")

In [ ]:
site_meta_filter = site_meta[site_meta.primary_veg_type.isin(veg_types)]

In [ ]:
site_meta_filter = gpd.GeoDataFrame(
    site_meta_filter,
    geometry=gpd.points_from_xy(site_meta_filter["lon"], site_meta_filter["lat"]),
    crs=4326
)

In [ ]:
site_meta_filter.explore()

In [ ]:
usa = gpd.read_file("data_in/ne_110m_admin_1_states_provinces.shp").query("iso_a2 == 'US' and name != 'Alaska' and name != 'Hawaii'").union_all()

In [ ]:
site_in_usa = site_meta_filter[site_meta_filter.within(usa)].copy()

How well do these sites explore climate space in CONUS?

In [ ]:
import xarray as xr
import rioxarray

In [ ]:
mat = xr.open_dataset("data_in/TerraClimate_19912020_tmax.nc").mean(dim="time").squeeze()
ppt = xr.open_dataset("data_in/TerraClimate_19912020_ppt.nc").sum(dim="time").squeeze()

In [ ]:
climate_space = xr.Dataset({
    "mat": mat.tmax,
    "ppt": ppt.ppt
}).rio.write_crs(4326)

In [ ]:
climate_space_clip = climate_space.rio.clip_box(*usa.bounds)

In [ ]:
phenocam_x = xr.DataArray(site_in_usa["lon"], dims=["points"])
phenocam_y = xr.DataArray(site_in_usa["lat"], dims=["points"])

phenocam_climate_space = climate_space_clip.sel(lon=phenocam_x, lat=phenocam_y, method="nearest").compute()

site_in_usa["ppt"] = phenocam_climate_space.ppt.data
site_in_usa["mat"] = phenocam_climate_space.mat.data
site_in_usa["site_years"] = (pd.to_datetime(site_in_usa["date_end"]) - pd.to_datetime(site_in_usa["date_start"])) / pd.Timedelta("365 days")

In [ ]:
fig, ax = plt.subplots(subplot_kw=dict(projection="scatter_density"))

tmax_arr = climate_space_clip.mat.data.flatten()
ppt_arr  = climate_space_clip.ppt.data.flatten()

mask = np.isnan(tmax_arr) | np.isnan(ppt_arr)
tmax_arr = tmax_arr[~mask]
ppt_arr  = ppt_arr[~mask]

cmap = plt.cm.viridis.copy()
cmap.set_under(alpha=0)

d = ax.scatter_density(tmax_arr, ppt_arr, norm=matplotlib.colors.Normalize(vmin=1, clip=False), cmap=cmap)

site_in_usa.plot.scatter(x="mat", y="ppt", s="site_years", ax=ax, color="red", alpha=0.7)

ax.set_xlabel("Mean annual Tmax (C)")
ax.set_ylabel("Mean annual ppt (mm)")
ax.set_title("")
plt.colorbar(d, label="Grid cells per pixel")
plt.grid()
plt.show()

In [ ]:
site_in_usa["primary_veg_type"] = site_in_usa["primary_veg_type"].map({
    "DB": "Deciduous forest",
    "GR": "Grassland",
    "EN": "Evergreen forest",
    "SH": "Shrubs",
    "EB": "Evergreen forest",
    "UN": "Forest understory"
})

site_in_usa["primary_veg_type"].value_counts()

In [ ]:
site_in_usa.groupby("primary_veg_type").agg({"site_years": "sum"})["site_years"].round()

In [ ]:
site_in_usa.iloc[0]